In [2]:
!pip install stomp.py

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.7 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=a217ac63bc6ab20093a5b26230100c86c9697a4b5ce2c80d57de9751dcfc574a
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [3]:
import time
import sys
import stomp

class MsgListener(stomp.ConnectionListener):
    def on_error(self, message):
        print('received an error "%s"' % message.body)
    def on_message(self, frame):
        print('received a message "%s"' % frame.body)

In [9]:
import asyncio
import json
import uuid

from websockets import connect, ConnectionClosed
from stomp.utils import Frame, convert_frame, parse_frame


async def send_frame(websocket, cmd, headers):
    """ Helper function for sending a frame """
    frame = Frame(cmd, headers=headers)
    await websocket.send(b''.join(convert_frame(frame)))


async def receive_message(websocket):
    """ Helper function to receive a message and parse it """
    message = await websocket.recv()
    return parse_frame(message)


async def connect_stomp(websocket, credentials): #domain, login, passcode):
    """ Connect to the STOMP server """
    await send_frame(websocket, 'CONNECT', credentials) #{'domain': domain, 'login': login, 'passcode': passcode})
    frame = await receive_message(websocket)
    if frame.cmd != 'CONNECTED':
        raise ConnectionRefusedError(f"STOMP authentication failed; {frame.headers.get('message', 'No error message')}")
    return json.loads(frame.body.decode('utf8').strip('\0'))


async def subscribe_to_topic(websocket, destination, selector):
    """ Subscribe to a topic on the STOMP server """
    subscribe_header = {
        'id': str(uuid.uuid4()),
        'destination': destination,
        'selector': selector,
    }
    await send_frame(websocket, 'SUBSCRIBE', subscribe_header)


async def main(url, credentials):
    async with connect(url, subprotocols=['STOMP']) as websocket:
        try:
            # Connect to STOMP with authentication
            metadata = await connect_stomp(websocket, credentials) #domain, login, passcode)
            print("Connected with metadata:", metadata)

            # Subscribe to the desired topic
            #await subscribe_to_topic(websocket, 'MXSE.orderbooks', 'TICKER="MXSE.TQBR.SBER"') # - это стакан реал-тайм
            await subscribe_to_topic(websocket, 'MXSE.candles', 'TICKER="MXSE.TQBR.SBER"') # - это свечки реал-тайм

            print("Subscription successful")

            # Main message processing loop
            while True:
                frame = await receive_message(websocket)
                if frame.cmd == 'MESSAGE':
                    body = json.loads(frame.body.decode('utf8').strip('\0'))
                    print(body)

        except ConnectionClosed:
            print("Connection closed unexpectedly!")
            return  # Exit the main function if connection is closed
        except Exception as e:
            print(f"An error occurred: {e}")
            return  # Handle other exceptions gracefully

In [11]:
if __name__ == '__main__':
    url = 'ws://iss.moex.com/infocx/v3/websocket'

    # Credentials
    credentials = {
        'domain': 'DEMO',
        'login': 'guest',
        'passcode': 'guest'
    }

    #asyncio.run(main(url, credentials)) # - вот это не работает в Colab
    await main(url, credentials)

CancelledError: 